# 02 - Interaction layers

Everything a channel post accumulates once it is published: comments underneath
it, reactions on it, poll votes inside it, service events around it, and the
language and topic it is written in. These are the layers a one-pass snapshot
corpus cannot record, and they are what the revisit design buys.

**Figures produced here**

| Output | Thesis figure |
|---|---|
| `anchoring_rate_dist` | comments-on-posts vs in-group chat, per group |
| `post_toggle_per_channel` | per-post comment toggling within channels |
| `thread_depth_dist` | reply-thread depth |
| `reaction_negativity_flag_rates` | Telegram flag rates by reaction-negativity decile |
| `poll_turnout` | poll turnout, counts and ratio |
| `service_actions` | service-action breakdown |
| `language_distribution` | channel-language distribution vs the predecessor |
| `topic_shares` | BERTopic topics over English channels |

**Tables produced here**: the comment-coverage decomposition, the comment-toggle
overview, and poll turnout by poll type.

Sections marked *offline* read a CSV from `../data/`; the rest need
`TGDATASET2_DB_URL`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from _shared import DATA, FIGURES, csv, q, setup_plots, shape

setup_plots()

## 1. What counts as a comment (*offline*)

Telegram does not give a message a flag saying "this is a comment on a post".
What it gives is a reply pointer, and the corpus has to reconstruct the rest.

The mechanism: when a broadcast channel has a linked discussion group, Telegram
auto-mirrors each post into that group as a forward from the channel itself. A
reader who comments on the post is really replying to that mirrored copy. So a
message is a comment on a post exactly when its reply chain roots in a mirrored
copy of a post, and it is in-group chat otherwise.

The `is_comment` column in the schema is not a reliable substitute; the
decomposition below is what the thesis uses.

In [ ]:
SQL_COMMENT_COVERAGE = """
    WITH linked_groups AS (
        -- A row with linked_to_channel_id set IS the discussion group; the value
        -- points at the parent broadcast channel.
        SELECT channel_id FROM channels WHERE linked_to_channel_id IS NOT NULL
    )
    SELECT
        count(*)                                                                     AS total,
        count(*) FILTER (WHERE reply_to_msg_id IS NOT NULL)                          AS has_reply_to,
        count(*) FILTER (WHERE reply_to_msg_id IS NOT NULL
                          AND channel_id IN (SELECT channel_id FROM linked_groups))  AS reply_in_linked_group,
        count(*) FILTER (WHERE reply_to_msg_id IS NOT NULL
                          AND reply_to_top_id IS NOT NULL
                          AND reply_to_top_id <> reply_to_msg_id)                    AS nested_reply,
        count(*) FILTER (WHERE reply_to_msg_id IS NOT NULL
                          AND reply_to_top_id IS NULL)                               AS top_level_reply,
        count(*) FILTER (WHERE is_comment)                                           AS is_comment_flag
    FROM messages
"""

csv("comment_coverage.csv", index_col=0)

### Anchoring rate per discussion group (*offline*)

For each linked group, what share of its replies actually root in a mirrored
post? A group at 1.0 is a pure comment section; a group near 0 is a chat room
that happens to be attached to a channel.

The query below identifies the mirrored copies (`dups`) with three conditions
together: the row is a forward, it comes from the parent channel, and its author
*is* the parent channel. The last clause matters. Without it, the roughly 0.5 %
of rows where a member manually re-forwards the parent's post would be counted as
mirrors, and replies to those would be miscounted as comments.

In [ ]:
SQL_ANCHORING = """
    WITH chats AS (
        SELECT channel_id AS chat_id, linked_to_channel_id AS parent
        FROM channels WHERE linked_to_channel_id IS NOT NULL
    ),
    m AS MATERIALIZED (
        SELECT m.channel_id, m.message_id, m.is_forwarded, m.fwd_from_channel_id,
               m.author_id, m.reply_to_msg_id, m.reply_to_top_id, c.parent
        FROM chats c JOIN messages m ON m.channel_id = c.chat_id
    ),
    dups AS (  -- channel-authored auto-mirrors only
        SELECT channel_id, message_id FROM m
        WHERE is_forwarded AND fwd_from_channel_id = parent AND author_id = parent
    ),
    flagged AS (
        SELECT r.channel_id,
               (dt.message_id IS NOT NULL OR tt.message_id IS NOT NULL) AS rooted
        FROM (SELECT channel_id, reply_to_msg_id, reply_to_top_id FROM m
              WHERE reply_to_msg_id IS NOT NULL) r
        LEFT JOIN dups dt ON dt.channel_id = r.channel_id AND dt.message_id = r.reply_to_msg_id
        LEFT JOIN dups tt ON tt.channel_id = r.channel_id AND tt.message_id = r.reply_to_top_id
    )
    SELECT channel_id, count(*) AS replies, count(*) FILTER (WHERE rooted) AS comments
    FROM flagged GROUP BY channel_id HAVING count(*) > 0
"""

per_group = csv("comment_anchoring_per_group.csv")
per_group["comment_frac"] = per_group["comments"] / per_group["replies"]
pooled = per_group["comments"].sum() / per_group["replies"].sum()
median = per_group["comment_frac"].median()
print(f"linked groups with replies : {len(per_group):,}")
print(f"replies (pooled)           : {int(per_group['replies'].sum()):,}")
print(f"comments on posts (pooled) : {int(per_group['comments'].sum()):,}  "
      f"({pooled:.1%} reply-weighted)")
print(f"per-group median           : {median:.1%}    mean: {per_group['comment_frac'].mean():.1%}")

fig, ax = plt.subplots(figsize=(5.48, 2.74))
ax.hist(per_group["comment_frac"].clip(0, 1), bins=40, color="#5b8ef0", edgecolor="white")
ax.axvline(median, ls="--", color="#d4584f", label=f"per-group median {median:.0%}")
ax.axvline(pooled, ls=":", color="#2e7d32", label=f"reply-weighted {pooled:.0%}")
ax.set_xlabel("Share of a group's replies anchored on a parent-duplicate (= comments)")
ax.set_ylabel("Linked groups")
ax.set_title("Comments-on-posts vs. in-group chat, per discussion group")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "anchoring_rate_dist.png", bbox_inches="tight")
fig.savefig(FIGURES / "anchoring_rate_dist.pdf", bbox_inches="tight")
plt.show()
print("Left tail = chat-style groups; the mode near 1.0 = pure comment sections.")

## 2. Who can comment at all (*offline*)

Comments are a per-post setting, not a per-channel one. `replies_count IS NULL`
on a post means the channel had comments switched off when it published; a
non-null value (even zero) means the discussion thread exists.

Two views: the corpus-wide split across all posts on channels that own a
discussion group, and the per-channel distribution of how often each channel
leaves comments on. The spikes at 0 and 1 in the second are channels that set the
policy once and never touched it; the mass in between is genuine per-post
toggling, which is a decision made post by post about which content is open to
reply.

In [ ]:
SQL_TOGGLE_OVERVIEW = """
    WITH parents AS (
        SELECT DISTINCT linked_to_channel_id AS parent
        FROM channels WHERE linked_to_channel_id IS NOT NULL
    )
    SELECT count(*)                                      AS posts,
           count(*) FILTER (WHERE replies_count IS NULL) AS disabled_null,
           count(*) FILTER (WHERE replies_count = 0)     AS enabled_empty,
           count(*) FILTER (WHERE replies_count > 0)     AS enabled_with
    FROM messages m
    WHERE m.channel_id IN (SELECT parent FROM parents)
      AND m.service_action IS NULL
"""

SQL_TOGGLE_PER_CHANNEL = """
    WITH parents AS (
        SELECT DISTINCT linked_to_channel_id AS parent
        FROM channels WHERE linked_to_channel_id IS NOT NULL
    )
    SELECT m.channel_id,
           count(*)                                            AS posts,
           count(*) FILTER (WHERE m.replies_count IS NOT NULL) AS enabled,
           count(*) FILTER (WHERE m.replies_count IS NULL)     AS disabled
    FROM messages m
    WHERE m.channel_id IN (SELECT parent FROM parents)
      AND m.service_action IS NULL
    GROUP BY m.channel_id
    HAVING count(*) >= 50
"""

display(csv("post_toggle_overview.csv", index_col=0))

per_parent = csv("post_toggle_per_channel.csv")
n = len(per_parent)
all_off = int((per_parent["enabled"] == 0).sum())
all_on = int((per_parent["disabled"] == 0).sum())
mixed = int(((per_parent["enabled"] > 0) & (per_parent["disabled"] > 0)).sum())
print(f"broadcast parents with >=50 non-service posts: {n:,}")
print(f"  all posts comments-OFF : {all_off:,}  ({all_off / n:.1%})")
print(f"  all posts comments-ON  : {all_on:,}  ({all_on / n:.1%})")
print(f"  MIXED (both present)   : {mixed:,}  ({mixed / n:.1%})")

fig, ax = plt.subplots(figsize=(5.48, 2.44))
ax.hist(per_parent["enabled_frac"].clip(0, 1), bins=40, color="#5b8ef0", edgecolor="white")
med = per_parent["enabled_frac"].median()
ax.axvline(med, ls="--", color="#d4584f", label=f"median {med:.2f}")
ax.set_xlabel("Share of a channel's posts with comments enabled (replies_count NOT NULL)")
ax.set_ylabel("Broadcast parents")
ax.set_title("Per-post comment toggling within channels that own a discussion group")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "post_toggle_per_channel.png", bbox_inches="tight")
fig.savefig(FIGURES / "post_toggle_per_channel.pdf", bbox_inches="tight")
plt.show()

## 3. How deep the threads go

Depth is measured by walking each message's `reply_to_msg_id` back to the thread
anchor and counting hops. Depth 0 is the anchor itself, 1 is a direct comment on
a post, 2 and beyond are replies to replies.

Measured on a single large discussion group rather than corpus-wide: the walk is
a per-message pointer chase, and doing it across every group would mean pulling
the whole reply graph. One deep group shows the shape; the distribution is
strongly concentrated at depth 1 in every group examined.

No CSV of the depth histogram was kept, so this section needs the database.

In [ ]:
CHAT = 1437657755   # a large linked discussion group, used as the depth showcase

SQL_THREAD = """
    SELECT message_id, reply_to_msg_id, reply_to_top_id
    FROM messages WHERE channel_id = :ch ORDER BY message_id
"""

chat_msgs = q(SQL_THREAD, ch=CHAT)
print(f"chat messages loaded: {len(chat_msgs):,}")

mid = chat_msgs["message_id"].to_numpy()
parent_map = dict(zip(mid.tolist(), chat_msgs["reply_to_msg_id"].to_numpy().tolist()))

def depth(m: int, max_hops: int = 50) -> int:
    """Hops from message m up to its thread anchor.

    Bounded by max_hops and guarded against cycles: a reply pointer can dangle
    (the parent may be outside the pulled range) or, rarely, loop.
    """
    d, cur, seen = 0, parent_map.get(m), {m}
    while cur is not None and not pd.isna(cur) and d < max_hops:
        if cur in seen:
            break
        seen.add(cur)
        d += 1
        cur = parent_map.get(int(cur))
    return d

depths = np.array([depth(int(m)) for m in mid])
depth_hist = pd.Series(depths).value_counts().sort_index()
display(depth_hist.head(15).to_frame("messages"))

fig, ax = plt.subplots(figsize=(5.48, 2.74))
trunc = depth_hist[depth_hist.index <= 15]
ax.bar(trunc.index, trunc.values, color="#4c78a8")
ax.set_yscale("log")
ax.set_xlabel("Depth from thread anchor (parent hops)")
ax.set_ylabel("Messages (log)")
ax.set_title("Reply-thread depth distribution")
fig.tight_layout()
fig.savefig(FIGURES / "thread_depth_dist.png", bbox_inches="tight")
fig.savefig(FIGURES / "thread_depth_dist.pdf", bbox_inches="tight")
plt.show()

## 4. Reactions as an audience signal (*offline*)

Reactions are a cheap, per-message audience signal that no snapshot corpus keeps.
The question here is whether they carry information Telegram's own moderation
flags do not already have.

Each channel gets a negativity score, `neg / (pos + neg)` over a fixed
language-neutral emoji lexicon, restricted to channels with at least 500 signal
reactions so the ratio is not noise. Channels are then split into deciles of that
score and compared against the platform flags on their latest metadata snapshot.

The lexicon is deliberately conservative: ambiguous emoji are in neither list.

In [ ]:
POSITIVE = ["\U0001F44D", "\u2764", "\u2764\uFE0F", "\U0001F525", "\U0001F602",
            "\U0001F389", "\U0001F4AF", "\U0001F970", "\U0001F60D", "\U0001F44F"]
NEGATIVE = ["\U0001F44E", "\U0001F4A9", "\U0001F921", "\U0001F92E", "\U0001F624",
            "\U0001F92C", "\U0001F621", "\U0001F622", "\U0001F494"]
THRESHOLD = 500   # ignore channels with too few signal reactions

SQL_NEGATIVITY = """
    WITH unrolled AS (
        SELECT m.channel_id, (r->>'emoticon') AS emoji, (r->>'count')::int AS cnt
        FROM messages m, LATERAL jsonb_array_elements(m.reactions) AS r
        WHERE m.reactions IS NOT NULL AND r ? 'emoticon'
    )
    SELECT channel_id,
           sum(cnt) FILTER (WHERE emoji = ANY(:pos)) AS pos,
           sum(cnt) FILTER (WHERE emoji = ANY(:neg)) AS neg,
           sum(cnt)                                  AS total
    FROM unrolled
    GROUP BY channel_id
"""

SQL_FLAGS = """
    SELECT DISTINCT ON (channel_id) channel_id, title, username,
           is_scam, is_verified, is_fake, is_restricted, subscriber_count
    FROM metadata_snapshots
    WHERE channel_id = ANY(:ids)
    ORDER BY channel_id, captured_at DESC
"""

deciles = csv("reaction_negativity_deciles.csv", index_col=0)
display(deciles)

fig, ax = plt.subplots(figsize=(5.48, 2.44))
x = np.arange(len(deciles))
w = 0.22
ax.bar(x - 1.5 * w, deciles["scam_rate"],       w, color="#d4584f", label="scam")
ax.bar(x - 0.5 * w, deciles["verified_rate"],   w, color="#54a24b", label="verified")
ax.bar(x + 0.5 * w, deciles["fake_rate"],       w, color="#f0a868", label="fake")
ax.bar(x + 1.5 * w, deciles["restricted_rate"], w, color="#8a6db2", label="restricted")
ax.set_xticks(x, labels=deciles.index.astype(str))
ax.set_xlabel("Negativity-score decile (D1 lowest)")
ax.set_ylabel("Flag rate")
ax.set_title("Telegram flag rates by reaction-negativity decile")
ax.legend(loc="upper left", fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES / "reaction_negativity_flag_rates.png", bbox_inches="tight")
fig.savefig(FIGURES / "reaction_negativity_flag_rates.pdf", bbox_inches="tight")
plt.show()

## 5. Polls

Polls are the one place where a channel asks its audience a question and Telegram
publishes the tally. The corpus captures the option vector, the quiz solution
where present, and `total_voters`.

Turnout has two readings. The **count** is how many people voted, which mostly
tracks channel size. The **ratio**, voters over views, is the share of the
audience that engaged, which is the interesting one.

Two confounders, both stated rather than corrected: `total_voters` is a single
capture-time snapshot with no vote accrual, and `views` is a separate lifetime
snapshot, so the two are taken at different moments. In a discussion group,
members can vote without inflating broadcast views, which pushes the ratio above
1. The central-tendency view clips at 1 and the share above 1 is reported
separately.

The per-poll turnout frame was not shipped (it is one row per poll), so the figure
needs the database. The summary table by poll type reads offline.

In [ ]:
SQL_POLLS = """
    SELECT channel_id, message_id, poll_id, question, answers,
           is_closed, is_quiz, is_multiple, is_public_voters,
           solution, total_voters, captured_at
    FROM polls
"""

SQL_TURNOUT = """
    SELECT p.channel_id, p.message_id, p.total_voters,
           m.views, m.date::date AS post_date
    FROM polls p
    JOIN messages m USING (channel_id, message_id)
"""

polls = q(SQL_POLLS)
turnout = q(SQL_TURNOUT)

# Attach the poll-side flags. Both frames carry total_voters from the same
# source, so the poll-side copy is renamed before the merge and used only to fill
# gaps; assigning the merged frame to a new name keeps the cell re-runnable.
poll_flags = polls[["channel_id", "message_id", "is_quiz", "is_public_voters", "total_voters"]]
turnout = turnout.merge(poll_flags.rename(columns={"total_voters": "tv_poll"}),
                        on=["channel_id", "message_id"], how="left")
turnout["total_voters"] = turnout["total_voters"].fillna(turnout["tv_poll"])

n_tv = int((turnout["total_voters"] > 0).sum())
print(f"polls with total_voters > 0: {n_tv:,} ({n_tv / len(turnout):.1%} of all polls)")

In [ ]:
tr = turnout[(turnout["views"] > 0) & (turnout["total_voters"] > 0)].copy()
tr["turnout_ratio"] = tr["total_voters"] / tr["views"]
over1 = float((tr["turnout_ratio"] > 1).mean())
trc = tr.copy()
trc["turnout_ratio"] = trc["turnout_ratio"].clip(upper=1.0)
print(f"polls with both views > 0 and voters > 0: {len(tr):,}")
print(f"  median turnout ratio (clipped at 1): {trc['turnout_ratio'].median():.3f}")
print(f"  share with ratio > 1 (group-poll / stale-views artefact): {over1:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(5.48, 1.74))
pos = turnout.loc[turnout["total_voters"] > 0, "total_voters"]
axes[0].hist(np.log10(pos), bins=45, color="#4c78a8")
axes[0].set_xlabel("log10(total_voters)")
axes[0].set_ylabel("polls")
axes[0].set_title(f"Poll turnout (counts), n={len(pos):,}")
for qz, color, label in [(False, "#4c78a8", "poll"), (True, "#e1a33a", "quiz")]:
    sub = trc.loc[trc["is_quiz"] == qz, "turnout_ratio"]
    if len(sub):
        axes[1].hist(sub, bins=30, alpha=0.6, color=color, label=label, density=True)
axes[1].set_xlabel("turnout ratio = voters / views (clipped at 1)")
axes[1].set_ylabel("density")
axes[1].set_title("Turnout ratio: quiz vs opinion")
# Legend inside, anchored at the centre line so it clears the clipped 1.0 bar.
axes[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout()
fig.savefig(FIGURES / "poll_turnout.png", bbox_inches="tight")
fig.savefig(FIGURES / "poll_turnout.pdf", bbox_inches="tight")
plt.show()

display(csv("poll_turnout_by_type.csv", index_col=0))

## 6. Service actions (*offline*)

Service messages are the structural events in a channel's life: members joining
and leaving, pins, title and photo changes, migrations, monetisation features
being switched on. The scraper stores them in `service_action` with their payload,
and they are a signal the predecessor corpus does not have at all.

Log scale on the x-axis: joins outnumber every other action by orders of
magnitude, and a linear axis would flatten the rest to nothing.

In [ ]:
SQL_SERVICE = """
    SELECT channel_id, message_id, date,
           service_action::text      AS action,
           service_action_data::text AS payload,
           is_pinned
    FROM messages
    WHERE service_action IS NOT NULL
"""

sa = csv("service_actions.csv")
print(f"distinct service_action values present: {len(sa)}")
print(f"total service messages: {sa['n'].sum():,}")

fig, ax = plt.subplots(figsize=(5.48, max(2.44, 0.17 * len(sa))))
ax.barh(sa["service_action"], sa["n"], color="#5b8ef0")
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("Messages (log)")
ax.set_title(f"Service-action breakdown: {sa['n'].sum():,} messages")
for i, v in enumerate(sa["n"]):
    ax.text(v, i, f"  {v:,}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "service_actions.png", bbox_inches="tight")
fig.savefig(FIGURES / "service_actions.pdf", bbox_inches="tight")
plt.show()
sa

## 7. Language and topic (*offline*)

Two composition checks against the predecessor corpus, both reported with N
because the absolute shares differ by construction: this is a different and much
smaller channel set. What is comparable is which languages and topics dominate,
and by how much.

**Language.** Every message carries a detected language in `message_languages`. A
channel is assigned a language when one language holds a strict majority of its
detected messages; channels without a majority are left undetermined and dropped
from the denominator, matching the predecessor's rule.

**Topic.** Topic assignments come from a BERTopic model fitted over the English
channels, one document per channel. The model run is a frozen input here: this
notebook reads the resulting topic table rather than refitting, which is why the
environment needs no transformer stack. Shares are over non-outlier channels,
since the predecessor's taxonomy has no outlier bucket. The dashed reference
lines mark the predecessor's three largest topic shares.

In [ ]:
SQL_CHANNEL_LANG = """
    WITH per AS (
        SELECT channel_id, lang, count(*) AS n,
               sum(count(*)) OVER (PARTITION BY channel_id) AS tot
        FROM message_languages
        GROUP BY channel_id, lang
    )
    SELECT DISTINCT ON (p.channel_id)
           p.channel_id, p.lang AS top_lang,
           p.n::float / p.tot AS top_share, p.tot AS n_detected,
           c.source, c.creation_date
    FROM per p
    JOIN channels c USING (channel_id)
    ORDER BY p.channel_id, p.n DESC
"""
# A channel is assigned its top language only when top_share > 0.5; the rest are
# "undetermined" and excluded from the denominator.

PAPER = {"ru": 0.355, "en": 0.163, "fa": 0.139, "de": 0.041, "ar": 0.021}
LANG_NAME = {"ru": "Russian", "en": "English", "fa": "Farsi",
             "de": "German", "ar": "Arabic"}

dist = csv("language_distribution.csv", index_col=0)
keys = ["ru", "en", "fa", "de", "ar"]
ours = [float(dist.loc[k, "our_share"]) if k in dist.index else 0.0 for k in keys]
papr = [PAPER[k] for k in keys]

fig, ax = plt.subplots(figsize=(5.48, 2.88))
x = np.arange(len(keys))
w = 0.38
ax.bar(x - w / 2, papr, w, label="TGDataset (KDD '25)", color="#9aa0a6")
ax.bar(x + w / 2, ours, w, label="ours (subset)", color="#3b7dd8")
ax.set_xticks(x)
ax.set_xticklabels([LANG_NAME[k] for k in keys])
ax.set_ylabel("share of assigned channels")
ax.set_title("Channel-language distribution: ours vs TGDataset (>50% rule)")
ax.legend(fontsize=8)
for xi, v in zip(x - w / 2, papr):
    ax.text(xi, v, f"{v:.0%}", ha="center", va="bottom", fontsize=7)
for xi, v in zip(x + w / 2, ours):
    ax.text(xi, v, f"{v:.0%}", ha="center", va="bottom", fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES / "language_distribution.png", bbox_inches="tight")
fig.savefig(FIGURES / "language_distribution.pdf", bbox_inches="tight")
plt.show()
dist

In [ ]:
topic_tbl = csv("topic_benchmark_table.csv", index_col=0)
n_modelled = int(topic_tbl["channels"].sum())
print(f"modelled {n_modelled:,} English channels into {len(topic_tbl)} topics "
      f"(outliers excluded from shares)")

fig, ax = plt.subplots(figsize=(5.48, 3.29))
tt = topic_tbl.head(15)
labels = [f"{t}: {str(kw)[:34]}" for t, kw in zip(tt.index, tt["keywords"])]
ax.barh(labels, tt["share"], color="#3b7dd8")
ax.invert_yaxis()
ax.set_xlabel("share of modelled English channels")
ax.set_title(f"BERTopic topics over English channels (n={n_modelled:,}): ours")
for s in (0.239, 0.149, 0.087):   # the predecessor's three largest topic shares
    ax.axvline(s, ls="--", color="#aaa", lw=0.8)
fig.tight_layout()
fig.savefig(FIGURES / "topic_shares.png", bbox_inches="tight")
fig.savefig(FIGURES / "topic_shares.pdf", bbox_inches="tight")
plt.show()
topic_tbl

## Reading

The comment layer is smaller than its raw row count suggests. A little under half
of all linked-group replies are anchored on a channel post; the rest is chat
happening in the same room. Quoting the row count of the linked-group layer as a
comment count overstates it by roughly a factor of three, which is why the
decomposition above is the number the thesis uses.

Comment availability is a live editorial choice, not a channel setting. A large
minority of channels that own a discussion group toggle comments post by post,
which means the presence of a comment thread is itself a signal about the content
under it.

The reaction-negativity deciles show a signal the platform flags do not carry:
the flag rates do not rise monotonically with audience negativity, so a channel
whose audience reacts badly is not thereby a channel Telegram has marked.

Language and topic composition track the predecessor's broad shape while
differing in level, which is the expected result for a smaller and more recent
channel set drawn by the same snowball method.